In [1]:
from sympy import *
from scipy.optimize import minimize
from scipy import integrate
from scipy.integrate import quad
from mpmath import nsum, inf

In [2]:
def E(peps,beta,r): #beta > 0
    g = lambda z1, z2, z3, k: 1/((exp(peps)-1)*(((z1+z2+z3-((exp(beta)/(exp(beta)-1))*(exp(beta*k)-1)))/exp(beta*k))**r)+exp(beta*r))

    m = lambda k: (1/exp(peps*k))*quad(lambda z1: quad(lambda z2: quad(lambda z3: g(z1,z2,z3,k), max(0,((exp(beta)/(exp(beta)-1))*(exp(beta*k)-1))-z1-z2), ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1))-z1-z2)[0], 0, ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1))-z1)[0], 0, ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1)))[0]
    M = nsum(lambda k: m(k), [0,inf])
    #print(M)

    c = lambda k: (1/exp(peps*k))*quad(lambda z1: quad(lambda z2: quad(lambda z3: (z1+z2+z3)*g(z1,z2,z3,k), max(0,((exp(beta)/(exp(beta)-1))*(exp(beta*k)-1))-z1-z2), ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1))-z1-z2)[0], 0, ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1))-z1)[0], 0, ((exp(beta)/(exp(beta)-1))*(exp(beta*(k+1))-1)))[0]
    C = nsum(lambda k: c(k), [0,inf])
    #print(C)
    
    return C/M

def E2(peps,beta,r): #beta = 0
    g = lambda z1, z2, z3, k: 1/((exp(peps)-1)*(((z1+z2+z3-k)/exp(beta*k))**r)+exp(beta*r))

    m = lambda k: (1/exp(peps*k))*quad(lambda z1: quad(lambda z2: quad(lambda z3: g(z1,z2,z3,k), max(0,k-z1-z2), k+1-z1-z2)[0], 0, k+1-z1)[0], 0, k+1)[0]
    M = nsum(lambda k: m(k), [0,inf])
    #print(M)

    c = lambda k: (1/exp(peps*k))*quad(lambda z1: quad(lambda z2: quad(lambda z3: (z1+z2+z3)*g(z1,z2,z3,k), max(0,k-z1-z2), k+1-z1-z2)[0], 0, k+1-z1)[0], 0, k+1)[0]
    C = nsum(lambda k: c(k), [0,inf])
    #print(C)
    
    return C/M

def g(z,beta,peps,r):
    return 1/((exp(peps)-1)*(z**r)+exp(beta*r))

def S(peps,beta,r):
    fun1 = lambda z: -g(z[0],beta,peps,r)/g(z[0]+1,beta,peps,r)
    bnds1 = ((0,exp(beta)-1), (0,None))
    res1 = minimize(fun1, (0,0), bounds = bnds1)
    #print(log(-res1.fun).evalf())
    fun2 = lambda z: -exp(peps)*g(z[0],beta,peps,r)/g((z[0]+1-exp(beta))/exp(beta),beta,peps,r)
    bnds2 = ((exp(beta)-0.999,exp(beta)), (0,None))
    res2 = minimize(fun2, (exp(beta)-1, 0), bounds = bnds2)
    #print(log(-res2.fun).evalf())
    x = max(-res1.fun, -res2.fun)
    return log(x)

In [3]:
eps = 5; beta = 0.2
#peps = x[0], r = x[1]

fun = lambda x: E(x[0],beta,x[1]) 
bnds = ((4*beta,eps), (0,None))
cons = ({'type': 'ineq', 'fun': lambda x:  eps-beta*3-S(x[0],beta,x[1])},)
res = minimize(fun, (eps,5), bounds = bnds, constraints = cons)
print(res.fun, res.x)

0.848894706115196 [4.41677658 4.96787586]
